# Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from xgboost import XGBClassifier as xgb
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import shap
import seaborn as sns
import lightgbm as lgb
from catboost import CatBoostClassifier
import matplotlib.pyplot as plt
import sys




In [ ]:
sys.path.insert(0,"..")
from src.data import load
# Download latest version
file_path = "uciml/default-of-credit-card-clients-dataset"
df = load.get_kaggle_data(file_path)


In [ ]:
print(df.head(), df.dtypes, df.isnull().sum())
df['PAY_0'].value_counts()

# EDA

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

print(y.value_counts())
y_negs = y[y==0].count()
y_pos = y[y==1].count()

weight_adjust = float(y_negs.sum()/y_pos.sum())
print(weight_adjust)

In [ ]:
sns.barplot(x=y.value_counts().index, y=y.value_counts().values)

In [ ]:
sns.heatmap(df.corr()[['target']])

#### Getting a better understanding of all the variables.

##### Payment Status

In [ ]:
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(pay_cols):
    groups = df.groupby(col)['target'].value_counts(normalize=True).unstack()
    groups.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'tomato'], legend=False)
    axes[i].set_title(col)
    axes[i].set_xlabel('Payment Status')
    axes[i].set_ylabel('Proportion')
    axes[i].tick_params(axis='x', rotation=0)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, ['No Default', 'Default'], loc='lower right', fontsize=12)
fig.suptitle('Default Rate by Payment Status (PAY_0 – PAY_6)', fontsize=14)
plt.tight_layout()
plt.show()


##### Limit Balance

In [ ]:
limit_cols = ['LIMIT_BAL']

fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=df, x='target', y='LIMIT_BAL', ax=ax, palette=['steelblue', 'tomato'])
ax.set_title('Limit Balance by Default Status')
ax.set_xlabel('Default')
ax.set_ylabel('Limit Balance')
plt.tight_layout()
plt.show()



##### Bill Amounts

In [ ]:
bill_cols = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']

fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharey=False)
axes = axes.flatten()

for i, col in enumerate(bill_cols):
    sns.boxplot(data=df, x='target', y=col, ax=axes[i], palette=['steelblue', 'tomato'])
    axes[i].set_title(col)
    axes[i].set_xlabel('Default')
    axes[i].set_ylabel('Amount')

fig.suptitle('Bill Amounts by Default Status (BILL_AMT1 – BILL_AMT6)', fontsize=14)
plt.tight_layout()
plt.show()


##### Demographics

In [ ]:
cat_cols = ['SEX', 'EDUCATION','MARRIAGE']

fig, axes = plt.subplots(1, 3, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    groups = df.groupby(col)['target'].value_counts(normalize=True).unstack()
    groups.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'tomato'], legend=False)
    axes[i].set_title(col)
    axes[i].set_xlabel(f'{col}')
    axes[i].set_ylabel('Proportion')
    axes[i].tick_params(axis='x', rotation=0)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, ['No Default', 'Default'], loc='lower right', fontsize=12)
fig.suptitle('Default Rate by Demographics', fontsize=14)
plt.tight_layout()
plt.show()

##### Education

In [ ]:
print(df['EDUCATION'].value_counts())
print(df.groupby('EDUCATION')['target'].value_counts(normalize=True).unstack())

#### Note: Combined 0,4,5,6 into 'Others' because these were the original counts and proportions

**EDUCATION category distribution (before cleanup)**

| EDUCATION | Count | Default Rate |
|-----------|-------|-------------|
| 0 (undocumented) | 14 | 0.0% |
| 1 (graduate school) | 10,585 | 19.2% |
| 2 (university) | 14,030 | 23.7% |
| 3 (high school) | 4,917 | 25.2% |
| 4 (other) | 123 | 5.7% |
| 5 (undocumented) | 280 | 6.4% |
| 6 (undocumented) | 51 | 15.7% |

Categories 0, 5, and 6 are undocumented and have small counts. Categories 0 and 5 show anomalously low default rates, likely due to data quality rather than genuine credit behavior. All three were merged into category 4 ("other") during preprocessing.

## Models

In [ ]:
from src.models import train, evaluate, explain

In [ ]:
X_train, X_test, y_train, y_test = train.split_data(df, 'target', 0.2)

### Gradient Boost

#### Get Base Model

In [ ]:
model = train.data_model(X_train, y_train, xgb(scale_pos_weight=weight_adjust, base_score=0.5, random_state=42))
base_vals = evaluate.evaluate_model(model, X_test, y_test, output_dict=True)
print(evaluate.evaluate_model(model, X_test, y_test, output_dict=True))

### Tune Hyperparameters

In [ ]:
hyperparam_tuning = RandomizedSearchCV(estimator=xgb(scale_pos_weight=weight_adjust, base_score=0.5, random_state=42),
                                       n_iter=10,
                                       param_distributions={'max_depth': [4,5,6],
                                                            'n_estimators': range(100,500),
                                                            'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3]},
                                       scoring='recall',
                                       random_state=42).fit(X_train, y_train)

In [ ]:
model_tuned = train.data_model(X_train, y_train, 
                               xgb(scale_pos_weight=weight_adjust, 
                                   base_score=0.5,
                                   random_state=42,
                                   max_depth=hyperparam_tuning.best_params_['max_depth'],
                                   n_estimators=hyperparam_tuning.best_params_['n_estimators'],
                                   learning_rate=hyperparam_tuning.best_params_['learning_rate']))
tuned_vals = evaluate.evaluate_model(model_tuned, X_test, y_test, output_dict=True)

print(base_vals, tuned_vals)

In [ ]:
print(hyperparam_tuning.best_params_)

##### Confusion Matrix

In [ ]:
y_pred_xgb = model_tuned.predict(X_test)
cm = confusion_matrix(y_test, y_pred_xgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Default','Default'])
disp.plot()
plt.show()

### Evaluate AUC-PR

In [ ]:
xgb_auc = evaluate.auc_eval(model_tuned, X_test, y_test)

### SHAP Values

In [ ]:
explain.shap_exp_gradient(X_test, model_tuned)

### LightGBM

#### Base Model

In [ ]:
lgb_model = train.data_model(X_train, y_train, lgb.LGBMClassifier(scale_pos_weight=weight_adjust, num_leaves=100, 
                                                                    max_depth=-1, learning_rate=0.1, n_estimators=250,
                                                                    random_state=42))
lgb_base_vals = evaluate.evaluate_model(lgb_model, X_test, y_test, output_dict=True)
print(evaluate.evaluate_model(lgb_model, X_test, y_test, output_dict=True))

### Hyperparameter Tuning

In [ ]:
lgb_hyperparam_tuning = RandomizedSearchCV(estimator=lgb.LGBMClassifier(scale_pos_weight=weight_adjust, random_state=42),
                                       n_iter=10,
                                       param_distributions={'max_depth': [4,5,6],
                                                            'n_estimators': range(100,500),
                                                            'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3]},
                                       scoring='recall',
                                       random_state=42).fit(X_train, y_train)
print(lgb_hyperparam_tuning.best_params_)

### Tune Model & Hyperparameters

In [ ]:
lgb_model_tuned = train.data_model(X_train, y_train, 
                               lgb.LGBMClassifier(scale_pos_weight=weight_adjust,
                                   random_state=42,
                                   max_depth=lgb_hyperparam_tuning.best_params_['max_depth'],
                                   n_estimators=lgb_hyperparam_tuning.best_params_['n_estimators'],
                                   learning_rate=lgb_hyperparam_tuning.best_params_['learning_rate']))
lgb_tuned_vals = evaluate.evaluate_model(lgb_model_tuned, X_test, y_test, output_dict=True)

#### Comparisons

##### LGB vs. Tuned LGB

In [ ]:
print(lgb_base_vals, lgb_tuned_vals)

##### Tuned XGB vs. Tuned LGB

In [ ]:
print(tuned_vals, lgb_tuned_vals)

##### Confusion Matrix

In [ ]:
y_pred_lgb = lgb_model_tuned.predict(X_test)
cm = confusion_matrix(y_test, y_pred_lgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Default','Default'])
disp.plot()
plt.show()

##### SHAP Explainability - LGB

In [ ]:
explain.shap_exp_gradient(X_test, lgb_model_tuned)

### AUC-PR Evaluation

In [ ]:
lgb_auc = evaluate.auc_eval(lgb_model_tuned, X_test, y_test)

## CatBoost

### Base Model

In [ ]:
cat_cols = ['SEX','MARRIAGE','EDUCATION']
cgb_model = train.data_model(X_train, y_train, 
                             CatBoostClassifier(
                                 scale_pos_weight=weight_adjust,
                                 random_seed=42,
                                 iterations=200, 
                                 learning_rate=0.1, 
                                 cat_features=cat_cols))
cgb_base_vals = evaluate.evaluate_model(cgb_model, X_test, y_test, output_dict=True)

In [ ]:
print(evaluate.evaluate_model(cgb_model, X_test, y_test, output_dict= True))

### Hyperparameter Tuning

In [ ]:
cgb_hyperparam_tuning = RandomizedSearchCV(estimator=CatBoostClassifier(scale_pos_weight=weight_adjust, random_seed=42),
                                       n_iter=10,
                                       param_distributions={'max_depth': [4,5,6],
                                                            'n_estimators': range(100,500),
                                                            'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3]},
                                       scoring='recall',
                                       random_state=42).fit(X_train, y_train)
print(cgb_hyperparam_tuning.best_params_)

In [ ]:
cgb_model_tuned = train.data_model(X_train, y_train, 
                               CatBoostClassifier(scale_pos_weight=weight_adjust,
                                   random_seed=42,
                                   max_depth=cgb_hyperparam_tuning.best_params_['max_depth'],
                                   n_estimators=cgb_hyperparam_tuning.best_params_['n_estimators'],
                                   learning_rate=cgb_hyperparam_tuning.best_params_['learning_rate']))
cgb_tuned_vals = evaluate.evaluate_model(cgb_model_tuned, X_test, y_test, output_dict=True)

In [ ]:
print(tuned_vals, lgb_tuned_vals, cgb_tuned_vals)

##### Confusion Matrix

In [ ]:
y_pred_cgb = cgb_model_tuned.predict(X_test)
cgb_cm = confusion_matrix(y_test, y_pred_cgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cgb_cm, display_labels=['No Default','Default'])
disp.plot()
plt.show()

#### Explainability: SHAP Values & AUC-PR

In [ ]:
import matplotlib.pyplot as plt
explainer = shap.TreeExplainer(cgb_model_tuned)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.savefig("../docs/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
cgb_auc = evaluate.auc_eval(cgb_model_tuned, X_test, y_test)
evaluate.auc_eval(cgb_model, X_test, y_test)


# Comparison Table

In [ ]:
summary = pd.DataFrame({
    'Model': ['XGBoost', 'LightGBM', 'CatBoost'],
    'Recall (Class 1)': [tuned_vals['1']['recall'], lgb_tuned_vals['1']['recall'], cgb_tuned_vals['1']['recall']],
    'Precision (Class 1)': [tuned_vals['1']['precision'], lgb_tuned_vals['1']['precision'], cgb_tuned_vals['1']['precision']],
    'F1 (Class 1)': [tuned_vals['1']['f1-score'], lgb_tuned_vals['1']['f1-score'], cgb_tuned_vals['1']['f1-score']],
    'AUC-PR': [xgb_auc[1], lgb_auc[1], cgb_auc[1]],
})

summary = summary.sort_values('Recall (Class 1)', ascending=False).reset_index(drop=True)
best_idx = summary['Recall (Class 1)'].idxmax()

def highlight_best(row):
    return ['background-color: #ffd700; color: black' if row.name == best_idx else '' for _ in row]

summary.style.apply(highlight_best, axis=1)